# OpenRouter OCR Batch Vote Extraction

This notebook processes every unique `doc_id` found in `ocr_table.csv` using an OpenRouter model.

For each `doc_id` it will:
- combine all OCR table fragments into one prompt
- constrain the result to party names from `submission_template.csv`
- validate the JSON locally
- save a per-document partial submission CSV immediately
- append a progress log entry for resumability

At the end it assembles one full submission file whose row count, order, and `id` values match the template exactly.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

from openai import OpenAI

print("openai version:", importlib.metadata.version("openai"))

In [ ]:
import csv
import difflib
import json
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

MODEL_NAME = ""
START_FROM_DOC_ID = ""
ONLY_DOC_IDS = []
OUTPUT_DIRNAME = "outputs"
BY_DOC_DIRNAME = "by_doc"
PROGRESS_LOG_NAME = "progress.jsonl"
FINAL_SUBMISSION_NAME = "final_submission.csv"


def guess_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "notebooks" / "election",
        Path.cwd().parent,
        Path.cwd().parent / "notebooks" / "election",
    ]
    for candidate in candidates:
        if (candidate / "ocr_table.csv").exists() and (
            candidate / "data" / "submission_template.csv"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the election notebook directory from the current working directory."
    )


BASE_DIR = guess_base_dir()
ENV_PATH = BASE_DIR / ".env"
OCR_TABLE_PATH = BASE_DIR / "ocr_table.csv"
SUBMISSION_TEMPLATE_PATH = BASE_DIR / "data" / "submission_template.csv"
OUTPUT_DIR = BASE_DIR / OUTPUT_DIRNAME
BY_DOC_DIR = OUTPUT_DIR / BY_DOC_DIRNAME
PROGRESS_LOG_PATH = OUTPUT_DIR / PROGRESS_LOG_NAME
FINAL_SUBMISSION_PATH = OUTPUT_DIR / FINAL_SUBMISSION_NAME


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(ENV_PATH)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL = os.environ.get(
    "OPENROUTER_MODEL", "meta-llama/llama-3.3-70b-instruct:free"
)
OPENROUTER_BASE_URL = os.environ.get(
    "OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"
)
OPENROUTER_SITE_URL = os.environ.get("OPENROUTER_SITE_URL", "")
OPENROUTER_APP_NAME = os.environ.get("OPENROUTER_APP_NAME", "Election OCR Extractor")
MODEL_NAME = MODEL_NAME or OPENROUTER_MODEL

print("BASE_DIR:", BASE_DIR)
print("ENV_PATH:", ENV_PATH)
print("OCR_TABLE_PATH:", OCR_TABLE_PATH)
print("SUBMISSION_TEMPLATE_PATH:", SUBMISSION_TEMPLATE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("BY_DOC_DIR:", BY_DOC_DIR)
print("PROGRESS_LOG_PATH:", PROGRESS_LOG_PATH)
print("FINAL_SUBMISSION_PATH:", FINAL_SUBMISSION_PATH)
print("OPENROUTER_API_KEY loaded:", bool(OPENROUTER_API_KEY))
print("MODEL_NAME:", MODEL_NAME)

In [ ]:
def read_csv_rows(path: Path) -> list[dict[str, str]]:
    csv.field_size_limit(10**9)
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def write_csv_rows(rows: list[dict], path: Path, fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def append_jsonl(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_template_rows(path: Path) -> list[dict[str, str]]:
    return read_csv_rows(path)


def group_template_rows_by_doc(
    rows: list[dict[str, str]],
) -> dict[str, list[dict[str, str]]]:
    grouped = {}
    for row in rows:
        grouped.setdefault(row["doc_id"], []).append(row)
    for doc_id in grouped:
        grouped[doc_id].sort(key=lambda row: int(row["row_num"]))
    return grouped


def group_ocr_rows_by_doc(path: Path) -> dict[str, list[dict[str, str]]]:
    grouped = {}
    for row in read_csv_rows(path):
        grouped.setdefault(row["id"], []).append(row)
    for doc_id in grouped:
        grouped[doc_id].sort(
            key=lambda row: (int(row["section_index"]), int(row["table_index"]))
        )
    return grouped


def combine_table_html(table_rows: list[dict[str, str]]) -> str:
    chunks = []
    for row in table_rows:
        chunks.append(
            "\n".join(
                [
                    f"<!-- doc_id={row['id']} -->",
                    f"<!-- section_index={row['section_index']} table_index={row['table_index']} tr_count={row['tr_count']} -->",
                    row["table_html"].strip(),
                ]
            )
        )
    return "\n\n".join(chunks)


def build_prompt(
    doc_id: str, allowed_parties: list[str], combined_tables_text: str
) -> str:
    allowed_text = "\n".join(
        f"- {index}. {party_name}"
        for index, party_name in enumerate(allowed_parties, start=1)
    )
    return f"""
You are extracting election vote counts from noisy OCR HTML tables.

Document id: {doc_id}

Return only a JSON array.
Do not add markdown fences.
Do not add explanations.
Do not add any keys besides `party_name` and `vote`.

Rules:
- Use only the allowed party names listed below.
- If OCR text is noisy, normalize it to the closest allowed party name.
- Return each party at most once.
- `vote` must be an integer in Arabic numerals only.
- Ignore parties that do not appear in the OCR tables.
- Ignore totals, summaries, headers, and non-party rows.
- The OCR may contain broken spacing, misspellings, <br>, <sup>, and malformed HTML.

Allowed party names:
{allowed_text}

Required output shape example:
[
  {{"party_name": "ประชาธิปัตย์", "vote": 10700}},
  {{"party_name": "ภูมิใจไทย", "vote": 127128}}
]

OCR HTML tables:
{combined_tables_text}
""".strip()


def openrouter_client() -> OpenAI:
    if not OPENROUTER_API_KEY.strip():
        raise ValueError(
            "Set OPENROUTER_API_KEY in notebooks/election/.env before calling OpenRouter."
        )
    return OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY.strip())


def extract_json_array(text: str):
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.startswith("json"):
            stripped = stripped[4:].strip()
    try:
        data = json.loads(stripped)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass
    start = stripped.find("[")
    end = stripped.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Model response does not contain a JSON array.")
    return json.loads(stripped[start : end + 1])


def call_openrouter(model_name: str, prompt: str):
    client = openrouter_client()
    extra_headers = {}
    if OPENROUTER_SITE_URL:
        extra_headers["HTTP-Referer"] = OPENROUTER_SITE_URL
    if OPENROUTER_APP_NAME:
        extra_headers["X-Title"] = OPENROUTER_APP_NAME
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": "You extract votes from OCR and return only valid JSON arrays.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        extra_headers=extra_headers or None,
    )
    message = completion.choices[0].message.content or ""
    if not message.strip():
        raise ValueError("OpenRouter returned an empty response body.")
    raw_rows = extract_json_array(message)
    actual_model = getattr(completion, "model", model_name)
    return completion, raw_rows, actual_model, message


def validate_model_rows(raw_rows: list[dict], allowed_parties: list[str]):
    if not isinstance(raw_rows, list):
        raise TypeError("Model output must be a JSON array.")
    allowed_set = set(allowed_parties)
    seen_parties = set()
    cleaned_rows = []
    issues = {
        "malformed_rows": [],
        "duplicate_party_names": [],
        "invalid_party_names": [],
        "invalid_votes": [],
    }
    for index, row in enumerate(raw_rows, start=1):
        if not isinstance(row, dict):
            issues["malformed_rows"].append({"index": index, "row": row})
            continue
        party_name = str(row.get("party_name", "")).strip()
        vote_value = row.get("vote")
        if party_name not in allowed_set:
            issues["invalid_party_names"].append(
                {
                    "index": index,
                    "party_name": party_name,
                    "suggestions": difflib.get_close_matches(
                        party_name, allowed_parties, n=3
                    ),
                }
            )
            continue
        if party_name in seen_parties:
            issues["duplicate_party_names"].append(
                {"index": index, "party_name": party_name}
            )
            continue
        try:
            vote_int = int(vote_value)
            if vote_int < 0:
                raise ValueError("vote must be non-negative")
        except (TypeError, ValueError) as exc:
            issues["invalid_votes"].append(
                {
                    "index": index,
                    "party_name": party_name,
                    "vote": vote_value,
                    "error": str(exc),
                }
            )
            continue
        seen_parties.add(party_name)
        cleaned_rows.append({"party_name": party_name, "vote": vote_int})
    return cleaned_rows, issues


def merge_submission_rows(
    template_rows: list[dict[str, str]], cleaned_rows: list[dict[str, int]]
):
    vote_by_party = {row["party_name"]: row["vote"] for row in cleaned_rows}
    merged_rows = []
    missing_parties = []
    for template_row in template_rows:
        party_name = template_row["party_name"]
        votes = vote_by_party.get(party_name, 0)
        if party_name not in vote_by_party:
            missing_parties.append(party_name)
        merged_rows.append(
            {
                "id": template_row["id"],
                "doc_id": template_row["doc_id"],
                "row_num": template_row["row_num"],
                "party_name": party_name,
                "votes": votes,
            }
        )
    return merged_rows, missing_parties


def by_doc_output_path(doc_id: str) -> Path:
    return BY_DOC_DIR / f"{doc_id}_submission.csv"


def load_progress_entries(path: Path) -> list[dict]:
    if not path.exists():
        return []
    entries = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            entries.append(json.loads(line))
    return entries


def latest_progress_by_doc(path: Path) -> dict[str, dict]:
    latest = {}
    for entry in load_progress_entries(path):
        latest[entry["doc_id"]] = entry
    return latest


def load_completed_doc_ids(progress_log_path: Path) -> set[str]:
    completed = set()
    for doc_id, entry in latest_progress_by_doc(progress_log_path).items():
        if entry.get("status") != "success":
            continue
        output_path = Path(entry.get("output_path", ""))
        if output_path.exists():
            completed.add(doc_id)
    return completed


def filter_doc_ids(
    doc_ids: list[str], start_from_doc_id: str, only_doc_ids: list[str]
) -> list[str]:
    selected = list(doc_ids)
    if only_doc_ids:
        only_set = set(only_doc_ids)
        selected = [doc_id for doc_id in selected if doc_id in only_set]
    if start_from_doc_id:
        selected = [doc_id for doc_id in selected if doc_id >= start_from_doc_id]
    return selected


def process_one_doc(
    doc_id: str, template_rows: list[dict[str, str]], table_rows: list[dict[str, str]]
):
    allowed_parties = [row["party_name"] for row in template_rows]
    combined_tables_text = combine_table_html(table_rows)
    prompt = build_prompt(doc_id, allowed_parties, combined_tables_text)
    completion, raw_rows, actual_model, raw_response_text = call_openrouter(
        MODEL_NAME, prompt
    )
    cleaned_rows, issues = validate_model_rows(raw_rows, allowed_parties)
    if any(issues.values()):
        raise ValueError(
            "Model output has validation issues: "
            + json.dumps(issues, ensure_ascii=False)
        )
    merged_rows, missing_parties = merge_submission_rows(template_rows, cleaned_rows)
    template_ids = [row["id"] for row in template_rows]
    merged_ids = [row["id"] for row in merged_rows]
    if len(merged_rows) != len(template_rows) or merged_ids != template_ids:
        raise ValueError(f"Merged rows do not match template ids for doc_id={doc_id}")
    return {
        "doc_id": doc_id,
        "actual_model": actual_model,
        "raw_response_text": raw_response_text,
        "raw_rows": raw_rows,
        "cleaned_rows": cleaned_rows,
        "issues": issues,
        "merged_rows": merged_rows,
        "missing_parties": missing_parties,
        "table_fragment_count": len(table_rows),
        "template_row_count": len(template_rows),
        "nonzero_votes": sum(1 for row in merged_rows if int(row["votes"]) != 0),
    }


def success_log_entry(result: dict, output_path: Path) -> dict:
    return {
        "timestamp": utc_now_iso(),
        "doc_id": result["doc_id"],
        "status": "success",
        "output_path": str(output_path),
        "model": result["actual_model"],
        "template_row_count": result["template_row_count"],
        "table_fragment_count": result["table_fragment_count"],
        "nonzero_votes": result["nonzero_votes"],
        "missing_parties_left_at_zero": len(result["missing_parties"]),
        "error": "",
    }


def failure_log_entry(doc_id: str, error: Exception) -> dict:
    return {
        "timestamp": utc_now_iso(),
        "doc_id": doc_id,
        "status": "failure",
        "output_path": "",
        "model": MODEL_NAME,
        "template_row_count": 0,
        "table_fragment_count": 0,
        "nonzero_votes": 0,
        "missing_parties_left_at_zero": 0,
        "error": repr(error),
    }


def assemble_final_submission(
    template_rows: list[dict[str, str]], completed_doc_ids: set[str]
) -> list[dict[str, str]]:
    votes_by_id = {}
    for doc_id in sorted(completed_doc_ids):
        path = by_doc_output_path(doc_id)
        if not path.exists():
            continue
        for row in read_csv_rows(path):
            votes_by_id[row["id"]] = str(row["votes"])
    final_rows = []
    for row in template_rows:
        final_rows.append(
            {
                "id": row["id"],
                "doc_id": row["doc_id"],
                "row_num": row["row_num"],
                "party_name": row["party_name"],
                "votes": votes_by_id.get(row["id"], row["votes"]),
            }
        )
    return final_rows


def assert_same_template_shape(
    final_rows: list[dict[str, str]], template_rows: list[dict[str, str]]
) -> None:
    if len(final_rows) != len(template_rows):
        raise AssertionError("Final submission row count does not match the template.")
    final_ids = [row["id"] for row in final_rows]
    template_ids = [row["id"] for row in template_rows]
    if final_ids != template_ids:
        raise AssertionError("Final submission id order does not match the template.")


TEMPLATE_ROWS = load_template_rows(SUBMISSION_TEMPLATE_PATH)
TEMPLATE_BY_DOC = group_template_rows_by_doc(TEMPLATE_ROWS)
OCR_BY_DOC = group_ocr_rows_by_doc(OCR_TABLE_PATH)
ALL_OCR_DOC_IDS = sorted(OCR_BY_DOC.keys())
ALL_TEMPLATE_DOC_IDS = sorted(TEMPLATE_BY_DOC.keys())
BATCH_DOC_IDS = filter_doc_ids(ALL_OCR_DOC_IDS, START_FROM_DOC_ID, ONLY_DOC_IDS)
NO_TABLE_DOC_IDS = sorted(set(ALL_TEMPLATE_DOC_IDS) - set(ALL_OCR_DOC_IDS))


In [ ]:
print(
    {
        "ocr_doc_ids": len(ALL_OCR_DOC_IDS),
        "template_doc_ids": len(ALL_TEMPLATE_DOC_IDS),
        "batch_doc_ids": len(BATCH_DOC_IDS),
        "template_docs_without_ocr_tables": len(NO_TABLE_DOC_IDS),
    }
)
print("First batch doc ids:", BATCH_DOC_IDS[:10])
print("First template doc ids without OCR tables:", NO_TABLE_DOC_IDS[:10])

In [ ]:
assert len(ALL_OCR_DOC_IDS) == 276
assert len(TEMPLATE_BY_DOC["party_list_10_24"]) == 57
assert len(OCR_BY_DOC["party_list_10_24"]) == 4
assert len(TEMPLATE_BY_DOC["constituency_10_1"]) > 0
assert len(OCR_BY_DOC["constituency_10_1"]) > 0
print("Sanity checks passed.")

In [ ]:
COMPLETED_DOC_IDS = load_completed_doc_ids(PROGRESS_LOG_PATH)
PENDING_DOC_IDS = [
    doc_id for doc_id in BATCH_DOC_IDS if doc_id not in COMPLETED_DOC_IDS
]

print(
    {
        "completed_doc_ids": len(COMPLETED_DOC_IDS),
        "pending_doc_ids": len(PENDING_DOC_IDS),
    }
)
print("First pending doc ids:", PENDING_DOC_IDS[:10])

In [ ]:
RUN_SUMMARY = []

for index, doc_id in enumerate(PENDING_DOC_IDS, start=1):
    print(f"[{index}/{len(PENDING_DOC_IDS)}] Processing {doc_id}")
    try:
        result = process_one_doc(doc_id, TEMPLATE_BY_DOC[doc_id], OCR_BY_DOC[doc_id])
        output_path = by_doc_output_path(doc_id)
        write_csv_rows(
            result["merged_rows"],
            output_path,
            ["id", "doc_id", "row_num", "party_name", "votes"],
        )
        entry = success_log_entry(result, output_path)
        append_jsonl(PROGRESS_LOG_PATH, entry)
        RUN_SUMMARY.append(entry)
        COMPLETED_DOC_IDS.add(doc_id)
        print(
            {
                "doc_id": doc_id,
                "model": result["actual_model"],
                "saved_rows": len(result["merged_rows"]),
                "nonzero_votes": result["nonzero_votes"],
                "missing_parties_left_at_zero": len(result["missing_parties"]),
            }
        )
    except Exception as exc:
        entry = failure_log_entry(doc_id, exc)
        append_jsonl(PROGRESS_LOG_PATH, entry)
        RUN_SUMMARY.append(entry)
        print("Stopping on first failure:", entry)
        raise

print("Batch processing complete for current pending docs.")

In [ ]:
FINAL_ROWS = assemble_final_submission(
    TEMPLATE_ROWS, load_completed_doc_ids(PROGRESS_LOG_PATH)
)
assert_same_template_shape(FINAL_ROWS, TEMPLATE_ROWS)
write_csv_rows(
    FINAL_ROWS,
    FINAL_SUBMISSION_PATH,
    ["id", "doc_id", "row_num", "party_name", "votes"],
)

print(
    {
        "final_submission_path": str(FINAL_SUBMISSION_PATH),
        "row_count": len(FINAL_ROWS),
        "completed_doc_ids_used": len(load_completed_doc_ids(PROGRESS_LOG_PATH)),
        "zero_vote_rows": sum(1 for row in FINAL_ROWS if int(row["votes"]) == 0),
    }
)

In [ ]:
FINAL_TEMPLATE_IDS = [row["id"] for row in FINAL_ROWS]
EXPECTED_TEMPLATE_IDS = [row["id"] for row in TEMPLATE_ROWS]

assert len(FINAL_ROWS) == len(TEMPLATE_ROWS)
assert FINAL_TEMPLATE_IDS == EXPECTED_TEMPLATE_IDS

PROGRESS_STATUS_COUNTS = Counter(
    entry["status"] for entry in load_progress_entries(PROGRESS_LOG_PATH)
)
print("Final submission matches template row count and id order.")
print("Progress status counts:", dict(PROGRESS_STATUS_COUNTS))